## Setup and Initialization

In [7]:
import sys
sys.path.insert(0, '/home/snt/projects_lujun/agi_index_tournament/src')

from agi_toolkit.DictRewriter import DictRewriter
import pandas as pd

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## Test 1: Initialize with Different Source Configurations

In [8]:
# Test 1a: Default configuration (WordNet + FastText)
print("\n=== Configuration 1: Default (WordNet + FastText) ===")
rewriter_default = DictRewriter()
print(f"Active sources: {rewriter_default.get_active_sources()}")
print(f"Available sources: {rewriter_default.get_available_sources()}")


=== Configuration 1: Default (WordNet + FastText) ===
✓ DictRewriter initialized with sources: ['wordnet', 'fasttext']
  Aggregation method: ranked
Active sources: ['wordnet', 'fasttext']
Available sources: ['wordnet', 'fasttext', 'sense2vec', 'fass']


In [9]:
# Test 1b: WordNet only
print("\n=== Configuration 2: WordNet Only ===")
rewriter_wn = DictRewriter(sources=['wordnet'])
print(f"Active sources: {rewriter_wn.get_active_sources()}")


=== Configuration 2: WordNet Only ===
✓ DictRewriter initialized with sources: ['wordnet']
  Aggregation method: ranked
Active sources: ['wordnet']


In [10]:
# Test 1c: Multiple sources with custom aggregation
print("\n=== Configuration 3: WordNet + FastText + FASS (Ranked Aggregation) ===")
rewriter_multi = DictRewriter(
    sources=['wordnet', 'fasttext', 'fass'],
    aggregation_method='ranked'
)
print(f"Active sources: {rewriter_multi.get_active_sources()}")
print(f"Aggregation method: {rewriter_multi.aggregation_method}")


=== Configuration 3: WordNet + FastText + FASS (Ranked Aggregation) ===
✓ DictRewriter initialized with sources: ['wordnet', 'fasttext', 'fass']
  Aggregation method: ranked
Active sources: ['wordnet', 'fasttext', 'fass']
Aggregation method: ranked


## Test 2: Word Sense Disambiguation with Multi-Source Synonyms

In [11]:
test_sentence = "The project develops powerful algorithms for machine learning applications."

print(f"\nInput: {test_sentence}\n")
print("WSD Results with Multi-Source Synonyms:")
print("=" * 100)

wsd_results = rewriter_default.wsd_sentence(test_sentence)

for item in wsd_results:
    if item['synonyms']:
        print(f"\nWord: {item['word']:12s} | POS: {item['pos']:4s}")
        print(f"  Primary synonyms (ranked): {', '.join(item['synonyms'][:5])}")
        print(f"  Definition: {item['gloss'][:60] if item['gloss'] else 'N/A'}...")
        if 'source_synonyms' in item and len(item['source_synonyms']) > 1:
            print(f"  Source breakdown:")
            for source_name, syns in item['source_synonyms'].items():
                if syns and source_name != 'ranked':
                    print(f"    - {source_name:10s}: {', '.join(syns[:3])}")


Input: The project develops powerful algorithms for machine learning applications.

WSD Results with Multi-Source Synonyms:

Word: project      | POS: NN  
  Primary synonyms (ranked): picture, stick out, see, cast, propose
  Definition: any piece of work that is undertaken or attempted...
  Source breakdown:
    - wordnet   : picture, stick out, see

Word: develops     | POS: VBZ 
  Primary synonyms (ranked): uprise, formulate, modernise, produce, rise
  Definition: make something new, such as a product or a mental or artisti...
  Source breakdown:
    - wordnet   : uprise, formulate, modernise

Word: powerful     | POS: JJ  
  Primary synonyms (ranked): sinewy, muscular, herculean, potent, mightily
  Definition: having great power or force or potency or effect...
  Source breakdown:
    - wordnet   : sinewy, muscular, herculean

Word: algorithms   | POS: NNS 
  Primary synonyms (ranked): algorithmic rule, algorithm, algorithmic program
  Definition: a precise rule (or set of rules) 

## Test 3: Rewriting with Different Sources

In [ ]:
test_text = "The project develops powerful algorithms for machine learning applications in modern technology."
ratio = 0.5

print(f"Original text:\n{test_text}\n")
print("=" * 100)

# Rewrite with WordNet only
print("\nRewritten with WordNet only:")
rewritten_wn, _ = rewriter_wn.rewrite(test_text, ratio=ratio)
print(f"{rewritten_wn}")

# Rewrite with default (WordNet + FastText)
print("\nRewritten with WordNet + FastText (Ranked):")
rewritten_default, _ = rewriter_default.rewrite(test_text, ratio=ratio)
print(f"{rewritten_default}")

## Test 4: Runtime Source Switching

In [ ]:
# Create a rewriter and dynamically change sources
rewriter = DictRewriter(sources=['wordnet'])
print(f"Initial sources: {rewriter.get_active_sources()}")

# Add more sources
rewriter.set_sources(['wordnet', 'fasttext'], aggregation_method='union')
print(f"Updated sources: {rewriter.get_active_sources()}")
print(f"Updated aggregation: {rewriter.aggregation_method}")

# Switch back
rewriter.set_sources(['wordnet'])
print(f"Reverted sources: {rewriter.get_active_sources()}")

## Test 5: Aggregation Strategy Comparison

In [ ]:
test_word = "good"
test_sentence = "This is a good project."

print(f"Test word: {test_word}")
print(f"Test sentence: {test_sentence}\n")

# Create rewriter with multiple sources
retriever = DictRewriter(sources=['wordnet', 'fasttext', 'fass'])

# Compare different aggregation methods
aggregation_methods = ['ranked', 'union', 'intersection', 'all']

for method in aggregation_methods:
    print(f"\nAggregation Method: {method}")
    print("-" * 60)
    wsd_results = retriever.wsd_sentence(test_sentence)
    for item in wsd_results:
        if item['word'].lower() == test_word:
            print(f"Synonyms: {item['synonyms'][:10]}")
            if 'source_synonyms' in item:
                for src, syns in item['source_synonyms'].items():
                    if syns:
                        print(f"  {src:15s}: {syns[:5]}")

## Test 6: Per-Call Source Override

In [ ]:
test_text = "The beautiful garden has many colorful flowers."
default_sources = ['wordnet']

rewriter = DictRewriter(sources=default_sources)
print(f"Base rewriter sources: {rewriter.get_active_sources()}\n")

# Rewrite with original sources
print("Rewrite 1 - Using base sources (WordNet):")
rewritten1, _ = rewriter.rewrite(test_text, ratio=0.4)
print(rewritten1)

# Rewrite with override sources (just this call)
print("\nRewrite 2 - Override with FastText only:")
rewritten2, _ = rewriter.rewrite(test_text, ratio=0.4, sources=['fasttext'])
print(rewritten2)

# Verify base rewriter is unchanged
print(f"\nBase rewriter sources after override: {rewriter.get_active_sources()}")

## Test 7: Custom FASS Dictionary

In [ ]:
# Note: To use custom FASS, create a JSON file with format:
# {"word": ["synonym1", "synonym2"], ...}

import json
import os

# Create example FASS dictionary
custom_fass = {
    "good": ["excellent", "wonderful", "outstanding"],
    "bad": ["poor", "terrible", "awful"],
    "big": ["large", "huge", "enormous"],
    "small": ["tiny", "little", "compact"],
    "fast": ["quick", "rapid", "swift"],
    "slow": ["sluggish", "gradual", "leisurely"]
}

# Save to file
fass_path = '/tmp/custom_fass.json'
with open(fass_path, 'w') as f:
    json.dump(custom_fass, f, indent=2)

print(f"Created custom FASS dictionary at: {fass_path}")
print(f"Content: {json.dumps(custom_fass, indent=2)}")

In [ ]:
# Load FASS and use in rewriting
rewriter_with_fass = DictRewriter(sources=['wordnet', 'fass'])
rewriter_with_fass.add_fass_dictionary(fass_path)

test_text = "This is a good and fast project."
print(f"Original: {test_text}")

rewritten, wsd_info = rewriter_with_fass.rewrite(test_text, ratio=1.0)
print(f"Rewritten: {rewritten}")

print("\nWord replacements:")
for word in wsd_info:
    if word['synonyms']:
        print(f"  {word['word']:10s} -> {word['synonyms'][:3]}")

## Test 8: Performance Comparison

In [ ]:
import time

test_text = "The project develops powerful algorithms for machine learning applications in modern technology and artificial intelligence systems."

configurations = [
    {'name': 'WordNet only', 'sources': ['wordnet']},
    {'name': 'FastText only', 'sources': ['fasttext']},
    {'name': 'WordNet + FastText', 'sources': ['wordnet', 'fasttext']},
]

results = []

for config in configurations:
    rewriter = DictRewriter(sources=config['sources'])
    
    start = time.time()
    rewritten, _ = rewriter.rewrite(test_text, ratio=0.5)
    elapsed = (time.time() - start) * 1000  # ms
    
    results.append({
        'Configuration': config['name'],
        'Sources': str(config['sources']),
        'Time (ms)': f"{elapsed:.2f}",
        'Words changed': sum(1 for w1, w2 in zip(test_text.split(), rewritten.split()) if w1 != w2)
    })

perf_df = pd.DataFrame(results)
print("Performance Comparison:")
print(perf_df.to_string(index=False))